# Phase 2a — PDF Downloader Demo

Walks through downloading one remote-URL PDF from the official_documents
DuckDB table, with checksum display + idempotency verification.

Per the multi-stage plan (see AGENTS.md). Phase 2b (the CocoIndex
pdf_to_markdown App) builds on the output of this notebook.

In [ ]:
# 1. Load the canonical DuckDB table to see the remote-URL rows.
import sqlite3, pathlib

DB_PATH = pathlib.Path("data/gemini_hackathon.duckdb")
print(f"DB exists: {DB_PATH.exists()}")

if DB_PATH.exists():
    with sqlite3.connect(str(DB_PATH)) as conn:
        rows = conn.execute(
            "SELECT source_key, subject, language, pdf_path "
            "FROM official_documents WHERE source_kind = 'remote_url'"
        ).fetchall()
    print(f"Remote-URL rows to download: {len(rows)}")
    for r in rows[:5]:
        print(f"  {r[0]:20s} {r[1]:20s} {r[2]:3s}  {r[3][:60]}")
else:
    print("Run `python -m dlt_pipelines.official_doc_fetcher` first to populate the DuckDB table.")

In [ ]:
# 2. Run the downloader + inspect the output.
from dlt_pipelines.pdf_downloader import run_downloader

stats = run_downloader()
print(stats)
# Expected on first run: {'considered': 7, 'downloaded': 7, 'skipped': 0, 'failed': 0}
# Expected on re-run:    {'considered': 0, 'downloaded': 0, 'skipped': 0, 'failed': 0}

In [ ]:
# 3. Verify the downloaded files on disk.
import pathlib, hashlib
from dlt_pipelines.pdf_downloader import PDF_RAW_ROOT

for pdf_path in sorted(PDF_RAW_ROOT.rglob("*.pdf")):
    size_kb = pdf_path.stat().st_size / 1024
    sha = hashlib.sha256(pdf_path.read_bytes()).hexdigest()[:16]
    print(f"{pdf_path.relative_to(PDF_RAW_ROOT.parent.parent):80s} {size_kb:8.1f} KiB  sha:{sha}...")

print(f"\nTotal: {sum(p.stat().st_size for p in PDF_RAW_ROOT.rglob('*.pdf')) / (1024*1024):.1f} MiB")

In [ ]:
# 4. Verify the DuckDB row state after the download.
if DB_PATH.exists():
    with sqlite3.connect(str(DB_PATH)) as conn:
        rows = conn.execute(
            "SELECT source_key, subject, language, source_kind, "
            "file_size_bytes, page_count, substr(sha256_hash, 1, 16) "
            "FROM official_documents "
            "WHERE source_kind = 'downloaded'"
        ).fetchall()
    print(f"Downloaded rows: {len(rows)}")
    for r in rows:
        print(f"  {r[0]:20s} {r[1]:20s} {r[2]:3s} -> {r[3]:10s} {r[4] or 0:6d} bytes  {r[5] or '?':>4} pages  sha:{r[6]}...")
else:
    print("No DB.")

## Summary

- All 7 remote-URL PDFs are now under `data/bi_ep/syllabi_raw/<source_key>/<subject>/<lang>/<sha256>.pdf`.
- The `official_documents` DuckDB table has `source_kind='downloaded'`, `pdf_path=<local-path>`, `sha256_hash=<full>`, `page_count=<n>`, `file_size_bytes=<n>` populated.
- Re-running the downloader is a no-op (considered=0).
- Ready for Phase 2b — the CocoIndex pdf_to_markdown App reads these files.